## Demo of client side Union of hotset dataset as view over Kafka through ISK and coldset dataset on MiniIO

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Jupyter").getOrCreate()

spark

25/10/08 12:17:09 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
spark.stop()

In [3]:
spark = None

In [4]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.remote("sc://localhost:15002").getOrCreate()


spark

In [5]:
spark.sql("use hot.isk").show()
spark.sql("show tables").show()


++
||
++
++

+---------+----------------+-----------+
|namespace|       tableName|isTemporary|
+---------+----------------+-----------+
|      isk|transactions_old|      false|
|      isk|        accounts|      false|
|      isk|       customers|      false|
|      isk|        branches|      false|
|      isk|    transactions|      false|
+---------+----------------+-----------+





### ISK view over Kafka has multiple topics including two topics with transactions data:
    
 - **transactions_old** with static set of data of 500000 events sorted by TransactionTime - that can be loaded into the cold set on MiniIO for demo purposes
 - **transactions** with dynamic set of data - starting with 50 events and gets new event every second.



In [ ]:
spark.sql("CREATE TABLE cold.data.index2 USING iceberg TBLPROPERTIES('format-version'='2') AS SELECT 0 as partition, 0 as offset;")

In [ ]:
%%sql

USE isk.isk

In [ ]:
%%sql

show tables;

In [ ]:
%%sql

DESCRIBE transactions_old;


In [ ]:
from datetime import datetime

In [ ]:
print(datetime.now())
spark.sql("SELECT * FROM transactions_old where transactionId='aaaaaaaa-aaaa-aaaa-aaaa-aaaaaaaaaaaaa';").show()
print(datetime.now())

In [ ]:
print(datetime.now())
spark.sql("SELECT * FROM transactions_old where transactionId='aaaaaaaa-aaaa-aaaa-aaaa-aaaaaaaaaaaaa' and kafka_partition=0 and kafka_offset=0;").show()
print(datetime.now())


In [ ]:
print(datetime.now())
spark.sql("SELECT * FROM transactions_old t join (SELECT 0 as partition, 0 as offset) index on (t.kafka_partition = index.partition and t.kafka_offset = index.offset) where t.transactionId='aaaaaaaa-aaaa-aaaa-aaaa-aaaaaaaaaaaaa';").show()
print(datetime.now())


In [ ]:
%%sql

SELECT 0 as kafka_partiton, 0 as kafka_offset 

In [ ]:
%%sql
--  Load transactions_old events into the coldset using CTAS operation, partitioned by TransactionTime hour.
CREATE TABLE minio.data.index2
          USING iceberg
          TBLPROPERTIES('format-version'='2')
 AS SELECT 0 as partition, 0 as offset;

In [ ]:
print(datetime.now())
spark.sql("SELECT * FROM transactions_old t join minio.data.index2 index on (t.kafka_partition = index.partition and t.kafka_offset = index.offset) where t.transactionId='aaaaaaaa-aaaa-aaaa-aaaa-aaaaaaaaaaaaa';").show()
print(datetime.now())


In [ ]:
#
# We've started in the middle, our hotset is too large and we must move it to cold storage
#

In [ ]:
%%sql
--  Load transactions_old events into the coldset using CTAS operation, partitioned by TransactionTime hour.
CREATE TABLE minio.data.transactions
          USING iceberg
          PARTITIONED BY (HOUR(TransactionTime))
          TBLPROPERTIES('format-version'='2')
 AS SELECT * FROM isk.isk.transactions_old;

In [ ]:
# Delete transactions_old from Kafka

In [ ]:
%%sql
-- Verify number of rows written to the cold set
select count (*) from  minio.data.transactions;

In [ ]:
# Show data now in minio

In [ ]:
#
# Now let's move on to the hotset, this shows incoming data from Kafka
#

In [ ]:
%%sql
-- Verify number of rows on the hot set in transactions topic
select count (*) from  isk.isk.transactions where transactionTime < Now();

In [ ]:
#
# We can union our hotset and coldset data to create the super powerful mixed applications
#

In [ ]:
%%sql
-- Count of transaction events across cold and hot set 

select count (*) from (SELECT * FROM minio.data.transactions m
UNION
select * from isk.isk.transactions i WHERE i.transactionTime> (SELECT max(transactionTime) FROM minio.data.transactions));

In [ ]:
# Run the count again to show new data seamlessly available

In [ ]:
#
# A more complex application
#

In [ ]:
%%sql
-- Total deposits - withdrawals per branch - without time bounds across cold and hotset   
    
select round(sum( 
case 
    when t.transactiontype='Withdrawal' THEN t.transactionamount*(-1)
    else t.transactionamount
END
),2) as total_per_branch,
b.branchname  
from 
(SELECT * FROM minio.data.transactions m
UNION
select * from isk.isk.transactions i) t    
    
    join branches b on t.branchid=b.branchid 
--    where t.transactiontime between '2025-04-21 00:00:00' AND '2025-04-21 23:59:59'
    group by b.branchname order by b.branchname asc;

In [ ]:
# Optional: show new data coming in

In [ ]:
%%sql
-- Latest 100 events on the hotset

select * from isk.isk.transactions ORDER BY transactionTime DESC LIMIT 100;